# 02 词表与 Token ID

上一节把文本切成了 token 序列。这一节继续回答：模型怎样给每个 token 分配一个可以用于查找的整数编号。

## 1. 为什么 Token 还要变成整数

即使文本已经被切成 token，神经网络仍然不能直接计算“我”“喜欢”这些符号。

因此 Tokenizer 会配合一个词表，把每个 token 映射到整数 ID：

$$
\text{Token 序列}
\rightarrow
\text{词表查找}
\rightarrow
\text{Token ID 序列}
$$

## 2. 什么是词表

词表 Vocabulary 是模型能够识别的一组 token，以及每个 token 对应的整数编号。

先构造一个很小的示例词表：

| Token | Token ID |
|---|---:|
| [PAD] | 0 |
| [UNK] | 1 |
| 我 | 2 |
| 喜欢 | 3 |
| 深度 | 4 |
| 学习 | 5 |

这个词表一共有 6 个 token，因此词表大小是：

$$
V=6
$$

## 3. 从 Token 序列到 ID 序列

按照上面的示例词表：

$$
[\text{我},\text{喜欢},\text{深度},\text{学习}]
\rightarrow
[2,3,4,5]
$$

这个过程通常称为编码 Encoding。

反过来，根据 ID 查回 token 的过程通常称为解码 Decoding：

$$
[2,3,4,5]
\rightarrow
[\text{我},\text{喜欢},\text{深度},\text{学习}]
$$

### 示意图：词表像一本“地址簿”

![Token 通过词表映射为整数 ID](assets/03_vocab_id_mapping.svg)

观察重点：词表负责建立稳定映射；ID 的用途是定位，不是描述语义。

## 4. Token ID 只是地址，不是特征

这是本节最重要的认识。

在示例词表中，“学习”的 ID 是 5，“我”的 ID 是 2，但这并不表示：

- “学习”比“我”大。
- “学习”的意义是“我”的 2.5 倍。
- ID 相近的 token 语义也相近。

Token ID 的作用类似书籍在书架上的编号。编号帮助我们找到目标位置，但编号本身不描述书的内容。

即使把词表重新编号，只要后续使用的映射也一起改变，模型仍然可以正常工作。

## 5. 为什么不能直接把 ID 当连续数值输入模型

假设三个 token 的 ID 分别是 10、11 和 900。

如果把 ID 本身当成普通数值，模型很容易误以为：

$$
10 \approx 11, \qquad 900 \gg 10
$$

但这些大小关系只是编号方式造成的，并不代表语义关系。

因此 Token ID 不能直接承担特征向量的职责。它更适合充当下一步 Embedding 查表操作的索引。

## 6. 遇到词表中不存在的内容怎么办

单词级词表容易遇到未登录词，也就是当前词表中不存在的新词。

常见处理思路有两种：

1. 用 [UNK] 代替无法识别的内容。
2. 使用子词或字符，把生僻内容拆成词表中已有的更小片段。

现代子词 Tokenizer 通常能显著减少整段内容只能变成 [UNK] 的情况。

## 7. Padding 与 Attention Mask

一个 batch 中的句子长度可能不同：

$$
\begin{aligned}
\text{句子 A} &: [2,3,5] \\
\text{句子 B} &: [2,3,4,5,6]
\end{aligned}
$$

为了组成规则的 Tensor，可以用 [PAD] 把短序列补齐。若 [PAD] 的 ID 是 0：

$$
\begin{aligned}
\text{句子 A} &: [2,3,5,0,0] \\
\text{句子 B} &: [2,3,4,5,6]
\end{aligned}
$$

同时生成 Attention Mask，告诉模型哪些位置是真实 token，哪些只是补齐位置：

$$
\begin{aligned}
\text{Mask A} &: [1,1,1,0,0] \\
\text{Mask B} &: [1,1,1,1,1]
\end{aligned}
$$

这里通常用 1 表示需要处理的位置，用 0 表示应忽略的补齐位置；具体接口的约定可能不同。

### 示意图：Padding 与 Attention Mask 如何配合

![短序列补齐后通过 Attention Mask 标记有效位置](assets/04_padding_attention_mask.svg)

观察重点：[PAD] 负责把形状补整齐，Mask 负责告诉模型哪些补齐位置不应被当作真实内容。

## 8. Token ID Tensor 的形状

设序列长度为 $L$，那么一个句子的 Token ID 可以组成长度为 $L$ 的一维序列。

一次处理 $B$ 个句子时，Token ID Tensor 的形状通常是：

$$
B \times L
$$

例如 batch size 为 2，统一后的序列长度为 5：

$$
\begin{bmatrix}
2 & 3 & 5 & 0 & 0 \\
2 & 3 & 4 & 5 & 6
\end{bmatrix}
\in \mathbb{N}^{2\times5}
$$

此时每个位置仍然只是一个整数索引，还没有变成 $D$ 维向量。

## 9. 到目前为止的数据变化

$$
\text{我喜欢深度学习}
\xrightarrow{\text{Tokenization}}
[\text{我},\text{喜欢},\text{深度},\text{学习}]
\xrightarrow{\text{词表编码}}
[2,3,4,5]
$$

现在模型已经得到了可以用于查表的地址。下一节会使用这些 ID，从 Embedding 矩阵中取出真正的向量。

## 10. 本节小结

1. 词表保存 token 与整数 ID 之间的映射。
2. Token ID 是索引或地址，不是语义特征。
3. ID 的大小和距离不能表示 token 的语义关系。
4. [UNK] 可以表示无法识别的内容，子词方法能减少这种情况。
5. [PAD] 用于补齐序列，Attention Mask 用于标记哪些位置应被忽略。
6. 一批 Token ID 的典型形状是 $B\times L$。

## 11. 自测问题

1. 词表保存了哪两种内容之间的映射？
2. 为什么 Token ID 不能表示语义大小？
3. 为什么不应该直接把整数 ID 当成连续数值特征？
4. [UNK] 和子词切分分别怎样处理生僻内容？
5. [PAD] 与 Attention Mask 为什么经常一起出现？
6. 如果 batch size 是 32，序列长度是 128，Token ID Tensor 的形状是什么？